# 🦙 Qwen con Ollama en Google Colab + Cloudflare Tunnel

Este notebook instala y ejecuta **Ollama** dentro de Google Colab, descarga el modelo **Qwen** deseado (`qwen2.5:7b`, `qwen2.5-coder:7b`, `qwen2.5:14b`, etc.) y crea un túnel público mediante **Cloudflare Tunnel**.

### 🌟 Ventajas:
- Interfaz y API idéntica a tu instalación local de Ollama (`/api/chat`, `/api/generate`, y endpoint `/v1` de OpenAI).
- Fácil cambio de modelos con comandos de Ollama (`ollama pull`, `ollama run`).

## 1. Verificar GPU

In [ ]:
#@title 🔍 Comprobar GPU
!nvidia-smi

## 2. Instalar Ollama y Cloudflared

In [ ]:
#@title 📦 Instalar Ollama y Cloudflared
!curl -fsSL https://ollama.com/install.sh | sh

!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1
!rm -f cloudflared-linux-amd64.deb
!pip install -q openai
print("✅ Ollama y Cloudflared instalados.")

## 3. Iniciar Servidor Ollama y Cloudflare Tunnel

In [ ]:
#@title 🚀 Iniciar Servicio Ollama y Túnel
import subprocess
import time
import re
import os

# Matar instancias previas si existen
!pkill -f ollama > /dev/null 2>&1
!pkill -f cloudflared > /dev/null 2>&1
time.sleep(1)

# Iniciar Ollama en segundo plano escuchando en todas las interfaces
ollama_env = os.environ.copy()
ollama_env["OLLAMA_HOST"] = "0.0.0.0:11434"
with open("ollama.log", "w") as ollama_log:
    subprocess.Popen("ollama serve", shell=True, env=ollama_env, stdout=ollama_log, stderr=ollama_log)

# Iniciar túnel de Cloudflare para el puerto 11434
with open("tunnel.log", "w") as tunnel_log:
    subprocess.Popen("cloudflared tunnel --url http://localhost:11434", shell=True, stdout=tunnel_log, stderr=tunnel_log)

print("⏳ Esperando que Ollama y Cloudflare inicien...")
public_url = None
for _ in range(25):
    time.sleep(2)
    try:
        with open("tunnel.log", "r") as f:
            content = f.read()
            match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", content)
            if match:
                public_url = match.group(0)
                break
    except Exception:
        pass

if public_url:
    print("=" * 65)
    print("🎉 ¡TÚNEL OLLAMA PÚBLICO DISPONIBLE!")
    print(f"🌐 Ollama Host / Base URL: {public_url}")
    print(f"🌐 OpenAI-compatible endpoint: {public_url}/v1")
    print("=" * 65)
else:
    print("⚠️ No se detectó la URL del túnel. Revisa tunnel.log")

## 4. Descargar Modelo Qwen
Elige el modelo Qwen deseado y descárgalo a la GPU de Colab.

In [ ]:
#@title 📥 Descargar Modelo con Ollama
MODEL_TAG = "qwen2.5-coder:7b" #@param ["qwen2.5:7b", "qwen2.5-coder:7b", "qwen2.5:14b", "qwen2.5-coder:14b", "qwen2.5:32b", "qwen2.5-coder:32b"]
!ollama pull {MODEL_TAG}
print(f"\n✅ Modelo {MODEL_TAG} listo para inferencia!")

## 5. Probar Inferencia con Cliente OpenAI

In [ ]:
#@title 🧪 Probar Endpoint con Python
from openai import OpenAI

client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")

response = client.chat.completions.create(
    model=MODEL_TAG,
    messages=[
        {"role": "user", "content": "Escribe un hola mundo en Rust con comentarios explicativos."}
    ],
    stream=True
)

for chunk in response:
    if chunk.choices and chunk.choices[0].delta.content:
        print(chunk.choices[0].delta.content, end="", flush=True)
print()